In [51]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import numpy as np


PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [52]:
# Configuration flags
USE_FRACTION: bool = False
SELECTED_SURFACE: str = "all"  # Options: "hard", "clay", or "all"
USE_EMBEDDING: bool = False

# Derived configuration
USE_FRACTION_STR: str = "ratio" if USE_FRACTION else "num"

In [53]:
# Base raw data directory
DATA_FOLDER = Path("../../../data")

# Subdirectories processed data
DATA_FOLDER_PROCESSED = DATA_FOLDER / "02_processed"
DATA_FOLDER_FINAL = DATA_FOLDER / "03_final"

# Processed file (depends on selected surface)
FILE_FEATURE  = DATA_FOLDER_PROCESSED / f"/feature__{USE_FRACTION_STR}_value__{SELECTED_SURFACE}.csv"

FILE_FINAL_TRAIN_ORIGINAL = DATA_FOLDER_FINAL / f"train_{SELECTED_SURFACE}_original.csv"
FILE_FINAL_VALID_ORIGINAL = DATA_FOLDER_FINAL / f"valid_{SELECTED_SURFACE}_original.csv"
FILE_FINAL_TEST_ORIGINAL  = DATA_FOLDER_FINAL / f"test_{SELECTED_SURFACE}_original.csv"

FILE_FINAL_TRAIN_DUPLICATE = DATA_FOLDER_FINAL / f"train_{SELECTED_SURFACE}_duplicate.csv"
FILE_FINAL_VALID_DUPLICATE = DATA_FOLDER_FINAL / f"valid_{SELECTED_SURFACE}_duplicate.csv"
FILE_FINAL_TEST_DUPLICATE  = DATA_FOLDER_FINAL / f"test_{SELECTED_SURFACE}_duplicate.csv"

# tmp file
FILE_TMP= DATA_FOLDER_PROCESSED / "tmp.csv"

In [ ]:
df = pd.read_csv(FILE_FEATURE, low_memory=False)

# Remove columns that start with 'Unnamed'
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["timestamp_year"] = df["timestamp"].dt.year
print(df.shape)

(117005, 2145)


## I. Describe columns

In [55]:
columns = [
# 'games_won_ratio_vs_opponent'
'money'
           ]


print(df[columns].describe())

              money
count  8.896100e+04
mean   3.559223e+06
std    5.643546e+06
min    2.100000e+05
25%    5.206240e+05
50%    9.750000e+05
75%    3.864414e+06
max    4.041280e+07


## II. Duplicate data

In [ ]:
from typing import Dict, List, Set

# Augmenting tennis match data by swapping Player 1 and Player 2
def augment_player_swap(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create an augmented version of a tennis match DataFrame by swapping all
    Player 1 and Player 2 columns, flipping difference values, inverting ratios,
    and adjusting winner labels.

    Parameters
    ----------
    df : pd.DataFrame
        Original tennis match dataset containing Player 1/2 features and match metadata.

    Returns
    -------
    pd.DataFrame
        Augmented DataFrame containing both original and swapped versions of each match.

    Notes
    -----
    - All columns starting with `p1_`/`p2_` or ending with `_p1`/`_p2` are automatically swapped.
    - Columns ending with `_diff` have their sign inverted.
    - Ratio-type columns (e.g., Player1/Player2 ratios) are inverted numerically.
    - The 'winner' column (if present) has its labels flipped (1 ↔ 2).
    - A new column `is_swapped` is added to distinguish original (0) from swapped (1) rows.
    """
    # Create a copy of the original DataFrame
    df_swapped = df.copy()

    # Identify column families
    p1_cols_start: List[str] = [c for c in df.columns if c.startswith("p1_")]
    p2_cols_start: List[str] = [c for c in df.columns if c.startswith("p2_")]
    p1_cols_end: List[str] = [c for c in df.columns if c.endswith("_p1")]
    p2_cols_end: List[str] = [c for c in df.columns if c.endswith("_p2")]
    diff_cols: List[str] = [c for c in df.columns if c.endswith("_diff")]

    # Compute matching suffix sets
    common_prefix_suffixes: Set[str] = {c.replace("p1_", "") for c in p1_cols_start} & {c.replace("p2_", "") for c in p2_cols_start}
    common_suffix_suffixes: Set[str] = {c.replace("_p1", "") for c in p1_cols_end} & {c.replace("_p2", "") for c in p2_cols_end}

    # Swap all p1_/p2_ columns
    for suffix in common_prefix_suffixes:
        p1_col, p2_col = f"p1_{suffix}", f"p2_{suffix}"
        if p1_col in df.columns and p2_col in df.columns:
            df_swapped[p1_col], df_swapped[p2_col] = df[p2_col], df[p1_col]

    # Swap all _p1/_p2 columns
    for suffix in common_suffix_suffixes:
        p1_col, p2_col = f"{suffix}_p1", f"{suffix}_p2"
        if p1_col in df.columns and p2_col in df.columns:
            df_swapped[p1_col], df_swapped[p2_col] = df[p2_col], df[p1_col]

    # Flip all *_diff columns
    for col in diff_cols:
        if col in df.columns:
            df_swapped[col] = -df[col]

    # Custom column pairs to swap explicitly
    special_cols: Dict[str, str] = {
        # Identification / categorical
        "player1_name": "player2_name",
        "player1_id": "player2_id",
        "player1_id_factor": "player2_id_factor",
        "player1_nationality": "player2_nationality",
        "player1_nationality_factor": "player2_nationality_factor",
        # Encoded features
        "hand_p1_encoded": "hand_p2_encoded",
        "backhand_p1_encoded": "backhand_p2_encoded",
    }

    for p1_col, p2_col in special_cols.items():
        if p1_col in df.columns and p2_col in df.columns:
            df_swapped[p1_col], df_swapped[p2_col] = df[p2_col], df[p1_col]

    # Handle ratio-type features by inversion
    ratio_cols: List[str] = [
        "games_won_ratio_vs_opponent",
        "tiebreak_ratio_vs_opponent",
    ]

    for col in ratio_cols:
        if col in df.columns:
            df_swapped[col] = (1 / df[col].replace(0, pd.NA)).fillna(0).astype(float)


    # Flip winner label (1 ↔ 2)
    if "winner" in df.columns:
        df_swapped["winner"] = df["winner"].replace({1: 2, 2: 1})

    # Add swap indicator column
    df = df.assign(is_swapped=0)
    df_swapped = df_swapped.assign(is_swapped=1)

    # Concatenate original and swapped DataFrames
    df_augmented = pd.concat([df, df_swapped], ignore_index=True)
    
    print("✅ Original:", df.shape)
    print("✅ Augmented:", df_augmented.shape)

    return df_augmented

In [ ]:
df = augment_player_swap(df=df)
df = df.sort_values(by=["timestamp"]).copy()

C:\Users\User\AppData\Local\Temp\ipykernel_2044\3978725690.py:84: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_swapped[col] = (1 / df[col].replace(0, pd.NA)).fillna(0)


✅ Original: (117005, 2146)
✅ Augmented: (234010, 2146)


## III. Analyse

In [94]:
def decide_zscore(df: pd.DataFrame, cols: list, threshold: float = 0.01) -> pd.DataFrame:
    """
    Analyse les colonnes et indique si un z-score est recommandé.
    
    Parameters
    ----------
    df : pd.DataFrame
        Jeu de données contenant les features.
    cols : list
        Liste des colonnes à tester.
    threshold : float
        Seuil de variance relative en-dessous duquel la standardisation n'est pas utile.
    
    Returns
    -------
    pd.DataFrame : tableau récapitulatif avec min, max et variance
    """
    decisions = []
    
    for col in cols:
        series = df[col].dropna()
        
        # Si la colonne n'est pas numérique, on ignore min/max/variance
        if pd.api.types.is_numeric_dtype(series):
            min_val, max_val = series.min(), series.max()
            variance = series.var()
        else:
            min_val, max_val, variance = None, None, None
        
        decisions.append([col, min_val, max_val, variance])
    
    return pd.DataFrame(decisions, columns=["Feature", "Min", "Max", "Variance"])

## IV. Output (y)

In [62]:
df['y'] = df['winner'].map({1: 0, 2: 1})

## V. Preprocessing

### A. Drop nan value

In [63]:
DROP_VALUE = ['type', 'money',
              "age_p1"                 ,"age_p2",
              # "height_p1"             ,"height_p2",
              "turn_pro_p1"            ,"turn_pro_p2", 
              "years_since_turn_pro_p1","years_since_turn_pro_p2",
              "hand_p1_encoded"        ,"hand_p2_encoded",
              "backhand_p1_encoded"    ,"backhand_p2_encoded",
              "p1_set1_score"          , "p2_set1_score",
              "p1_set2_score"          , "p2_set2_score"]

df = df.dropna(subset=DROP_VALUE)

In [64]:
missing = df.isnull().sum()
print("Missing values before cleaning:\n", missing[missing > 0])

Missing values before cleaning:
 global_duration                            82912
p1_set1_tiebreak                          155034
p1_set1_duration                           82932
p1_set2_tiebreak                          157301
p1_set2_duration                           82932
                                           ...  
under_odd_Unibet_full_GAMES_66.5_end      176802
over_odd_Unibet_full_GAMES_51.5_start     176802
over_odd_Unibet_full_GAMES_51.5_end       176802
under_odd_Unibet_full_GAMES_51.5_start    176802
under_odd_Unibet_full_GAMES_51.5_end      176802
Length: 1431, dtype: int64


### B. ONE HOT encoding

In [65]:
df_encoded = pd.get_dummies(df, columns=['surface_group'], prefix='surface')

### C. Clip

In [77]:
set_played_last_5_max = 25
set_played_last_10_max = 37

df["p1_total_sets_played_last_5"] = df["p1_total_sets_played_last_5"].clip(upper=set_played_last_5_max)
df["p2_total_sets_played_last_5"] = df["p2_total_sets_played_last_5"].clip(upper=set_played_last_5_max)

df["p1_total_sets_played_last_10"] = df["p1_total_sets_played_last_10"].clip(upper=set_played_last_10_max)
df["p2_total_sets_played_last_10"] = df["p2_total_sets_played_last_10"].clip(upper=set_played_last_10_max)

### D. Log1p

In [78]:
df_final = df.copy()
df_final = df_final.sort_values(by=["match_date", "match_id"])

In [79]:
log1p_counts = [
    "p1_total_games_won", "p2_total_games_won",
    "p1_total_sets_played", "p2_total_sets_played",
    "p1_tiebreaks_played", "p2_tiebreaks_played",
    "p1_wins_after_losing_first_set", "p2_wins_after_losing_first_set",
    "p1_losses_after_losing_first_set", "p2_losses_after_losing_first_set",
    "simple_power_p1", "simple_power_p2",
    "simple_exp_p1", "simple_exp_p2",
    "simple_log_p1", "simple_log_p2",
    "p1_avg_set_margin", "p2_avg_set_margin",

    "popularity_multiplicative_both_diff", 'popularity_additive_both_diff',
    'popularity_geometric_mean_both_diff', 'popularity_weighted_sum_both_diff',
    "games_won_ratio_vs_opponent", "tiebreak_ratio_vs_opponent",
    "money", "trueskill_diff", "glicko_diff", "elo_diff",
    "simple_power_diff", 'simple_exp_diff', 'simple_log_diff',

    "p1_trueskill_diff_last_5"     , "p2_trueskill_diff_last_5",
    "p1_glicko_diff_last_5"        , "p2_glicko_diff_last_5",
    "p1_elo_diff_last_5"           , "p2_elo_diff_last_5",
    "p1_total_sets_played_last_5"  , "p2_total_sets_played_last_5",
    "p1_days_since_last_5"         , "p2_days_since_last_5",

    "p1_trueskill_diff_last_10"    , "p2_trueskill_diff_last_10",
    "p1_glicko_diff_last_10"       , "p2_glicko_diff_last_10",
    "p1_elo_diff_last_10"          , "p2_elo_diff_last_10",
    "p1_total_sets_played_last_10" , "p2_total_sets_played_last_10",
    "p1_days_since_last_10"        , "p2_days_since_last_10",
]


for col in log1p_counts:
    if col in df_final.columns:
        df_final[col] = np.sign(df_final[col]) * np.log1p(np.abs(df_final[col]))


print("✅ log1p transformation applied on count-based columns.")

✅ log1p transformation applied on count-based columns.


## VI. Split (Train, Valid & Test)

In [80]:
df_filtered = df_final.copy()


In [81]:
# Minimum number of appearances required for a player to be kept
MIN_MATCHES = 11

# Display initial dataset shape
print(f"Initial dataset shape: {df_filtered.shape}")
initial_rows = df_filtered.shape[0]

# Count appearances for each player as player1
player1_counts = df_filtered["player1_id_factor"].value_counts()

# Identify valid players with enough appearances
valid_players = player1_counts[player1_counts >= MIN_MATCHES].index

#  Filter DataFrame to keep only rows with valid player1
df_filtered = df_filtered[df_filtered["player1_id_factor"].isin(valid_players)]

# Display results
filtered_rows = df_filtered.shape[0]
rows_removed = initial_rows - filtered_rows
percent_removed = 100 * rows_removed / initial_rows

print(f"Filtered dataset shape: {df_filtered.shape}")
print(f"Rows removed: {rows_removed} ({percent_removed:.2f}% reduction)")
print(f"Threshold applied: {MIN_MATCHES} appearances as player1")


Initial dataset shape: (176804, 2149)
Filtered dataset shape: (171337, 2149)
Rows removed: 5467 (3.09% reduction)
Threshold applied: 11 appearances as player1


In [82]:
df_final = df_filtered.copy()
df_final = df_final.sort_values(by=["match_date", "match_id"])

In [ ]:
# Define split size
train_size = 4_000

train_df = df_final.iloc[:-train_size].reset_index(drop=True)
valid_df = df_final.iloc[-train_size:].reset_index(drop=True)

# Remove from train any match_id that exists in validation
valid_match_ids = set(valid_df["match_id"].unique())
train_df = train_df[~train_df["match_id"].isin(valid_match_ids)].reset_index(drop=True)

print(f"Train shape after filtering: {train_df.shape}")
print(f"Validation shape: {valid_df.shape}")

common_ids = set(train_df["match_id"]).intersection(valid_df["match_id"])
print(f"Common match_ids remaining: {len(common_ids)}")  # should be 0

# train_df = train_df[train_df["is_swapped"] == 0]
# valid_df = valid_df[valid_df["is_swapped"] == 1]

valid_df["is_swapped"].value_counts()

Train shape after filtering: (167336, 2149)
Validation shape: (4000, 2149)
Common match_ids remaining: 0


is_swapped
0    2015
1    1985
Name: count, dtype: int64

## VII. Pipeline

In [85]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

### A. FEATURE GROUPS

In [ ]:
# Define the pairs of columns
pairs = [ # list of (p1_, p2_) column pairs
    ('age_p1'                          , 'age_p2'),
    ('years_since_turn_pro_p1'         , 'years_since_turn_pro_p2'),
    ('trueskill_p1'                    , 'trueskill_p2'),
    ('glicko_p1'                       , 'glicko_p2'),
    ('elo_p1'                          , 'elo_p2'),
    ('simple_exp_p1'                   , 'simple_exp_p2'),
    ('simple_log_p1'                   , 'simple_log_p2'),
    ('simple_power_p1'                 , 'simple_power_p2'),
    ('p1_pct_set_win_ratio'            , 'p2_pct_set_win_ratio'),
    ('p1_avg_set_margin'               , 'p2_avg_set_margin'),
    ('p1_total_games_won'              , 'p2_total_games_won'),
    ('p1_total_sets_played'            , 'p2_total_sets_played'),
    ('p1_tiebreaks_played'             , 'p2_tiebreaks_played'),
    ("p1_wins_after_losing_first_set"  , "p2_wins_after_losing_first_set"),
    ("p1_losses_after_losing_first_set", "p2_losses_after_losing_first_set"),
    ("p1_set_success_rate"             , "p2_set_success_rate"),
    ("p1_MVI"                          , "p2_MVI"),
    ("p1_elo_diff_last_5"              , "p2_elo_diff_last_5"),
    ("p1_trueskill_diff_last_5"         , "p2_trueskill_diff_last_5"),       
    ("p1_glicko_diff_last_5"            , "p2_glicko_diff_last_5"),        
    ("p1_total_sets_played_last_5"     , "p2_total_sets_played_last_5"),
    ("p1_total_games_lost_last_5"      , "p2_total_games_lost_last_5"),  
    ("p1_total_games_won_last_5"       , "p2_total_games_won_last_5"),
    ("p1_avg_games_won_per_set_last_5" , "p2_avg_games_won_per_set_last_5"),
    ("p1_days_since_last_5"            , "p2_days_since_last_5"),  
    ("p1_elo_diff_last_10"             , "p2_elo_diff_last_10"),        
    ("p1_trueskill_diff_last_10"        , "p2_trueskill_diff_last_10"),
    ("p1_glicko_diff_last_10"          , "p2_glicko_diff_last_10"),
    ("p1_total_sets_played_last_10"    , "p2_total_sets_played_last_10"),
    ("p1_total_games_lost_last_10"     , "p2_total_games_lost_last_10"),
    ("p1_total_games_won_last_10"      , "p2_total_games_won_last_10"),
    ("p1_avg_games_won_per_set_last_10", "p2_avg_games_won_per_set_last_10"),
    ("p1_days_since_last_10"           , "p2_days_since_last_10"),
]

z_score_cols = ['trueskill_diff', 'glicko_diff', 'elo_diff', ]
z_score_cols += ["money"]
z_score_cols +=['simple_exp_diff', 'simple_log_diff', 'simple_power_diff']

z_score_cols += ['popularity_multiplicative_both_diff', 'popularity_additive_both_diff',
                 'popularity_geometric_mean_both_diff', 'popularity_weighted_sum_both_diff',]


z_score_cols += ["games_won_ratio_vs_opponent" , "tiebreak_ratio_vs_opponent"]

# z_score_cols += ['past_match_short_logit', 'past_match_medium_logit', 'past_match_long_logit']
# z_score_cols += ['elo_short_logit', 'elo_medium_logit', 'elo_long_logit']
# z_score_cols += ['trueskill_short_logit', 'trueskill_medium_logit', 'trueskill_long_logit']
# z_score_cols += ['glicko_short_logit', 'glicko_medium_logit', 'glicko_long_logit']

columns_to_add  = ['y', "tournament_id", "is_swapped", "match_id"]
columns_to_add += ['player1_id_factor', 'player2_id_factor', 'player1_nationality_factor','player2_nationality_factor']
columns_to_add += ['p1_odd_home_away_Unibet_full_start'       , 'p2_odd_home_away_Unibet_full_start',
                   'p1_odd_home_away_Betclic_full_start'      , 'p2_odd_home_away_Betclic_full_start']

columns_to_add += ['hand_p1_encoded', 'hand_p2_encoded', 'backhand_p1_encoded', 'backhand_p2_encoded']
columns_to_add += ['p1_avg_games_won_per_set'         , 'p2_avg_games_won_per_set',
                   'p1_avg_games_lost_per_set'        , 'p2_avg_games_lost_per_set',
                   'p1_pct_set1_win_ratio'            , 'p2_pct_set1_win_ratio',
                   'p1_pct_tiebreak_win'              , 'p2_pct_tiebreak_win',
                   "p1_pct_win_after_losing_first_set", "p2_pct_win_after_losing_first_set",
                   "p1_pct_game_win"                  , "p2_pct_game_win",
                   ]


### B. INDIVIDUAL PIPELINES

In [87]:
from sklearn.base import BaseEstimator, TransformerMixin
class PairedColumnStandardScaler(BaseEstimator, TransformerMixin):
    """Custom transformer to apply Z-score normalization to paired columns."""

    def __init__(self, pairs):
        self.pairs = pairs
        self.scalers = {}

    def fit(self, X, y=None):
        """Fit the scaler to the data by calculating mean and standard deviation for each pair."""
        for pair in self.pairs:
            combined = np.concatenate([X[pair[0]].values, X[pair[1]].values])
            mean = np.mean(combined)
            scale = np.std(combined)
            self.scalers[pair] = (mean, scale)
        return self

    def transform(self, X):
        """Apply the Z-score normalization to the data."""
        X_transformed = X.copy()
        for pair in self.pairs:
            mean, scale = self.scalers[pair]
            for col in pair:
                X_transformed[col] = (X[col] - mean) / scale
        return X_transformed

    def fit_transform(self, X, y=None):
        """Fit to data, then transform it."""
        self.fit(X)
        return self.transform(X)
    
    def get_feature_names_out(self, input_features=None):
        """Get feature names for the output."""
        return [col for pair in self.pairs for col in pair]

In [88]:
class TimeFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        # No fitting needed
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        if X.shape[1] != 1:
            raise ValueError("Expected exactly one datetime column.")
        
        dt = pd.to_datetime(X.iloc[:, 0])
        numeric_time = dt.view("int64") // 10**9
        month = dt.dt.month
        dayofweek = dt.dt.dayofweek

        features = pd.DataFrame({
            "timestamp_numeric": numeric_time,
            "month_sin": np.sin(2 * np.pi * month / 12),
            "month_cos": np.cos(2 * np.pi * month / 12),
            "dow_sin": np.sin(2 * np.pi * dayofweek / 7),
            "dow_cos": np.cos(2 * np.pi * dayofweek / 7),
        }, index=X.index)

        return features
    
    def fit_transform(self, X, y=None):
        """Fit to data, then transform it."""
        self.fit(X)
        return self.transform(X)

    def get_feature_names_out(self, input_features=None):
        """Return the output feature names."""
        return np.array([
            "timestamp_numeric",
            "month_sin",
            "month_cos",
            "dow_sin",
            "dow_cos"
        ])

In [89]:
# --- 7.1 Match type (categorical → one-hot)
type_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='if_binary'))
])

# --- 7.2 Surface type (categorical → one-hot)
surface_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- 7.3 Round stage (ordinal + scale)
round_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['1', '2', '3', '4', '8', '16', '32', '64', '128']],
        handle_unknown='use_encoded_value', unknown_value=-1
    )),
    ('scale', StandardScaler())
])

# --- 7.4 Z-score numerical columns
zscore_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('scale', StandardScaler())
])

# --- 7.5 Paired numerical features
paired_scaler = PairedColumnStandardScaler(pairs)

# --- 7.6 Timestamp features
time_pipeline = TimeFeatureExtractor()

### C. COLUMN TRANSFORMER COMBINATION

In [90]:
preprocessor = ColumnTransformer(transformers=[
    ('type'         , type_pipeline    , ['type']),
    ('surface'      , surface_pipeline , ['surface']),
    ('round'        , round_pipeline   , ['round']),
    ('paired_scaler', paired_scaler    , [col for pair in pairs for col in pair]),
    ('extras'       , zscore_pipeline  , z_score_cols),
    ('timestamp'    , time_pipeline    , ['timestamp']),
], remainder='drop')


### D. FIT AND TRANSFORM 

In [91]:
X_train_arr = preprocessor.fit_transform(train_df)
X_valid_arr = preprocessor.transform(valid_df)

feature_names = preprocessor.get_feature_names_out()

train_preprocessed = pd.DataFrame(X_train_arr, columns=feature_names, index=train_df.index)
valid_preprocessed = pd.DataFrame(X_valid_arr, columns=feature_names, index=valid_df.index)

train_preprocessed[columns_to_add] = train_df[columns_to_add].loc[train_preprocessed.index]
valid_preprocessed[columns_to_add] = valid_df[columns_to_add].loc[valid_preprocessed.index]


C:\Users\User\AppData\Local\Temp\ipykernel_2044\4223974744.py:16: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  numeric_time = dt.view("int64") // 10**9
C:\Users\User\AppData\Local\Temp\ipykernel_2044\4223974744.py:16: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  numeric_time = dt.view("int64") // 10**9


## VIII. Save

In [92]:
print(f"shape of train_preprocessed: {train_preprocessed.shape}")
print(f"shape of valid_preprocessed: {valid_preprocessed.shape}")

shape of train_preprocessed: (167336, 127)
shape of valid_preprocessed: (4000, 127)


In [ ]:
# Analyse
decide_zscore(df=train_preprocessed, cols=train_preprocessed.columns)

,Feature,Min,Max,Variance
0,type__type_250.0,0.0,1.0,0.249031
1,type__type_500.0,0.0,1.0,0.127579
2,type__type_750.0,0.0,1.0,0.005125
3,type__type_1000.0,0.0,1.0,0.165448
4,type__type_1500.0,0.0,1.0,0.004770
...,...,...,...,...
122,p2_pct_tiebreak_win,0.0,1.0,0.047988
123,p1_pct_win_after_losing_first_set,0.0,1.0,0.009324
124,p2_pct_win_after_losing_first_set,0.0,1.0,0.010227
125,p1_pct_game_win,0.0,1.0,0.017028


In [ ]:
train_preprocessed_duplicate = train_preprocessed.copy()
valid_preprocessed_duplicate = valid_preprocessed.copy()

train_preprocessed_original = train_preprocessed[train_preprocessed["is_swapped"] == 0].copy()
valid_preprocessed_original = valid_preprocessed[valid_preprocessed["is_swapped"] == 0].copy()

In [ ]:
train_preprocessed_duplicate.to_csv(FILE_FINAL_TRAIN_DUPLICATE)
valid_preprocessed_duplicate.to_csv(FILE_FINAL_VALID_DUPLICATE)
# test_preprocessed_duplicate.to_csv(FILE_FINAL_TEST_DUPLICATE)

train_preprocessed_original.to_csv(FILE_FINAL_TRAIN_ORIGINAL)
valid_preprocessed_original.to_csv(FILE_FINAL_VALID_ORIGINAL)
# test_preprocessed_original.to_csv(FILE_FINAL_TEST_ORIGINAL)

print(f"duplicate : {train_preprocessed_duplicate.shape} - {valid_preprocessed_duplicate.shape}")
print(f"original  : {train_preprocessed_original.shape}  - {valid_preprocessed_original.shape}")


duplicate : (167336, 127) - (4000, 127)
original  : (83818, 127)  - (2015, 127)


## IX. STAT

In [ ]:
# Get the minimum and maximum timestamps from the validation dataset
timestamp_min = valid_preprocessed_original["timestamp__timestamp_numeric"].min()
timestamp_max = valid_preprocessed_original["timestamp__timestamp_numeric"].max()

# Convert timestamps (in seconds) to datetime for better readability
start_date = pd.to_datetime(timestamp_min, unit='s')
end_date = pd.to_datetime(timestamp_max, unit='s')

# Compute duration in seconds, days, and approximate months
duration_seconds = timestamp_max - timestamp_min
duration_days = duration_seconds / (24 * 3600)
duration_months = duration_days / 30.44  # Average days per month (365.25 / 12)

# Display a readable summary of the validation period
print("📅 Validation period:")
print(f"  • Start date : {start_date.date()}")
print(f"  • End date   : {end_date.date()}")
print(f"  • Duration   : {duration_seconds:.0f} seconds "
      f"(≈ {duration_days:.1f} days ≈ {duration_months:.2f} months)")


📅 Validation period:
  • Start date : 2025-03-15
  • End date   : 2025-10-10
  • Duration   : 18021600 seconds (≈ 208.6 days ≈ 6.85 months)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Convert timestamps (in seconds) to datetime for clearer visualization
valid_preprocessed_original["timestamp_dt"] = pd.to_datetime(
    valid_preprocessed_original["timestamp__timestamp_numeric"], unit='s'
)

# Plot the temporal density of validation matches
plt.figure(figsize=(10, 5))
sns.kdeplot(
    data=valid_preprocessed_original,
    x="timestamp_dt",
    fill=True,
    color="steelblue",
    bw_adjust=0.5  # adjust the smoothing bandwidth if needed
)

# Customize the plot appearance
plt.title("Temporal Distribution of Validation Matches", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


NameError: name 'valid_preprocessed_original' is not defined